# Notebook 01 — Circuit Visualization & Expressibility Diagnostics

This notebook provides a visual and quantitative analysis of the two ansatz families
used in the hybrid VQC classifier:

- **StronglyEntanglingLayers** (SEL): parameterised Rot gates + CNOT ladders with arbitrary range
- **BasicEntanglingLayers** (BEL): simpler Ry + nearest-neighbour CNOT structure

**Key questions answered here:**
1. What do the circuits look like at different depths?
2. How does expressibility (Meyer–Wallach Q) scale with circuit depth?
3. Why does SEL mitigate barren plateaus better than a global ansatz at moderate qubit counts?

**References:**
- Cerezo et al. (2021). *Variational quantum algorithms.* Nature Reviews Physics. [arXiv:2012.09265](https://arxiv.org/abs/2012.09265)
- Sim et al. (2019). *Expressibility and entangling capability of PQCs.* [arXiv:1905.10876](https://arxiv.org/abs/1905.10876)
- Holmes et al. (2022). *Connecting ansatz expressibility to gradient magnitudes.* PRX Quantum. [arXiv:2101.02138](https://arxiv.org/abs/2101.02138)
- McClean et al. (2018). *Barren plateaus in quantum neural network training.* Nature Communications. [arXiv:1803.11173](https://arxiv.org/abs/1803.11173)

In [ ]:
import sys
from pathlib import Path

# Add src to path (works whether or not the package is installed)
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml

from qml_hybrid.circuits import (
    make_vqc_circuit,
    weight_shape,
    meyer_wallach_expressibility,
)

print(f"PennyLane version: {qml.__version__}")
np.random.seed(42)


## 1 — Circuit Diagrams

### 1a. StronglyEntanglingLayers (SEL)

SEL uses parameterised `Rot(φ, θ, ω)` gates (full SU(2)) followed by CNOT ladders
with *arbitrary range* (not just nearest-neighbour). This gives high expressibility
but more circuit depth.

The weight tensor shape is `(n_layers, n_qubits, 3)` — three rotation angles per qubit per layer.

### 1b. BasicEntanglingLayers (BEL)

BEL uses `RY(θ)` gates followed by nearest-neighbour CNOT rings. Simpler and shallower,
but lower expressibility. Weight shape: `(n_layers, n_qubits)`.

### Why not a Hardware-Efficient Ansatz (HEA)?

The two-qubit gate connectivity in BEL matches the topology of neutral-atom processors
(e.g. QuEra Aquila), where entanglement is mediated by Rydberg blockade between
nearest-neighbour atoms in a 1D or 2D register. SEL is less hardware-native but
maximises expressibility on the simulator, making it the better baseline for
establishing an accuracy ceiling before hardware-noise studies.

In [ ]:
N_QUBITS = 4  # Small for readable diagrams
N_LAYERS = 2

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, ansatz in zip(axes, ["strongly_entangling", "basic_entangling"]):
    circuit = make_vqc_circuit(N_QUBITS, N_LAYERS, ansatz=ansatz, embedding="angle")
    w = np.zeros(weight_shape(N_QUBITS, N_LAYERS, ansatz))
    x = np.zeros(N_QUBITS)

    # qml.draw_mpl returns (fig, ax) — draw into our axes
    draw_fig, draw_ax = qml.draw_mpl(circuit)(w, x)
    plt.close(draw_fig)

    # Re-render into our subplot
    ax.set_title(f"{ansatz}\n(n_qubits={N_QUBITS}, n_layers={N_LAYERS})", fontsize=10)
    ax.axis("off")

# Draw separately for better resolution
for ansatz in ["strongly_entangling", "basic_entangling"]:
    circuit = make_vqc_circuit(N_QUBITS, N_LAYERS, ansatz=ansatz, embedding="angle")
    w = np.zeros(weight_shape(N_QUBITS, N_LAYERS, ansatz))
    x = np.zeros(N_QUBITS)
    fig_c, ax_c = qml.draw_mpl(circuit)(w, x)
    ax_c.set_title(
        f"{ansatz} | n_qubits={N_QUBITS}, n_layers={N_LAYERS} | "
        f"params={np.prod(weight_shape(N_QUBITS, N_LAYERS, ansatz))}",
        fontsize=9,
    )
    plt.tight_layout()
    plt.show()


## 2 — Meyer–Wallach Expressibility

The **Meyer–Wallach measure** $Q$ quantifies the *average entanglement* generated
by a circuit over uniformly random parameters:

$$Q = \frac{2}{n} \mathbb{E}_{\boldsymbol{\theta}} \left[ \sum_{k=0}^{n-1} \left(1 - \mathrm{Tr}(\rho_k^2) \right) \right]$$

where $\rho_k = \mathrm{Tr}_{\neq k}(|\psi(\boldsymbol{\theta})\rangle\langle\psi(\boldsymbol{\theta})|)$
is the single-qubit reduced density matrix for qubit $k$.

- $Q = 0$: circuit generates no entanglement (product state).
- $Q = 1$: circuit generates maximal average entanglement.

High expressibility is desirable for learning, but comes at the cost of susceptibility
to **barren plateaus** (McClean et al., 2018), where gradient variance decays as
$\mathrm{Var}[\partial_\theta E] \sim \mathcal{O}(2^{-n})$ for deep global-cost circuits.
Using *local* cost functions (per-qubit PauliZ measurements, as done here) partially
mitigates this by keeping gradient variance polynomial for layerwise locality.

In [ ]:
from tqdm.auto import tqdm

N_QUBITS_EXP = 4
DEPTHS = [1, 2, 3, 4, 5]
N_SAMPLES = 100  # increase to 500 for publication-quality estimates

print("Computing Meyer-Wallach expressibility (this may take ~1-2 minutes)...")

mw_scores = {}
for ansatz in ["strongly_entangling", "basic_entangling"]:
    scores = []
    for depth in tqdm(DEPTHS, desc=ansatz):
        q = meyer_wallach_expressibility(
            n_qubits=N_QUBITS_EXP,
            n_layers=depth,
            ansatz=ansatz,
            n_samples=N_SAMPLES,
            seed=42,
        )
        scores.append(q)
        mw_scores[(ansatz, depth)] = q
    print(f"\n{ansatz}: {[f'{s:.3f}' for s in scores]}")

print("\nDone.")


In [ ]:
from qml_hybrid.utils import plot_expressibility

fig = plot_expressibility(mw_scores)
plt.show()

# Print summary table
import pandas as pd

rows = []
for (ansatz, depth), q in sorted(mw_scores.items()):
    n_params = int(np.prod(weight_shape(N_QUBITS_EXP, depth, ansatz)))
    rows.append({"ansatz": ansatz, "n_layers": depth, "n_params": n_params, "meyer_wallach_Q": f"{q:.4f}"})

df_exp = pd.DataFrame(rows)
print("\nExpressibility summary:")
print(df_exp.to_string(index=False))


## 3 — Ansatz Selection Rationale

Based on the expressibility curves above:

- **StronglyEntanglingLayers** (SEL) achieves higher $Q$ at matched depth because its
  arbitrary-range CNOT connections create long-range entanglement unavailable in the
  nearest-neighbour BEL topology.
- The $Q$ plateau for SEL at depth ≥3 suggests diminishing returns beyond 3 layers
  for 4-qubit circuits — consistent with the observation that the accessible Hilbert
  space is well-covered at that depth.

**Barren plateau mitigation strategy used in this project:**
1. *Local cost function*: measuring $\langle Z_k \rangle$ per qubit rather than a global
   observable keeps gradient variance $\mathcal{O}(1/\mathrm{poly}(n))$ instead of $\mathcal{O}(2^{-n})$.
2. *Moderate depth*: n_layers=3 is chosen as the point where SEL's $Q$ begins to plateau,
   avoiding excessive over-parameterisation.
3. *He-initialisation* of the classical layers with small-angle initialisation for the
   quantum weights prevents the circuit from starting in a high-symmetry point.

This analysis directly parallels hyperparameter sensitivity analysis in the candidate's
Bayesian optimisation work at ANL, where the goal is also to characterise the loss
landscape before committing to a specific architecture.